In [ ]:
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import datetime

# Fetch historical data from yfinance
def get_yfinance_data(symbol, interval, start_date, end_date):
    df = yf.download(symbol, interval=interval, start=start_date, end=end_date)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index.name = 'timestamp'
    return df

# Parameters
symbol = 'BTC-USD'
start_date = '2025-06-01'
end_date = '2025-06-30'
capital = 500  # Starting capital
risk_per_trade = 0.01  # 1% risk
commission = 0.001  # 0.1% fee per trade

# Fetch 5-minute and 1-hour data
data_5min = get_yfinance_data(symbol, '5m', start_date, end_date)
data_1h = get_yfinance_data(symbol, '1h', start_date, end_date)

# Prepare data: Get first 5-minute candle per hour
data_5min['hour'] = data_5min.index.floor('H')
first_5min = data_5min.groupby('hour').first()[['High', 'Low']]
first_5min.index.name = 'timestamp'

# Merge with 1-hour data
data_1h = data_1h.join(first_5min, how='left', rsuffix='_5min')
data_1h.dropna(inplace=True)

# Initialize tracking
trades = []
equity = [capital]
returns = []

# Backtest loop
for i in range(1, len(data_1h)):
    row = data_1h.iloc[i]
    prev_row = data_1h.iloc[i-1]
    high_5min = row['High_5min'].item()  # Extract scalar
    low_5min = row['Low_5min'].item()    # Extract scalar
    prev_close = prev_row['Close'].item() # Extract scalar
    entry_price = None
    stop_loss = None
    position_size = 0
    direction = None

    # Check for breakout
    if prev_close > high_5min:
        # Long entry
        entry_price = high_5min
        stop_loss = low_5min
        direction = 'long'
        risk = entry_price - stop_loss
        if risk > 0:
            position_size = (capital * risk_per_trade) / risk
    elif prev_close < low_5min:
        # Short entry
        entry_price = low_5min
        stop_loss = high_5min
        direction = 'short'
        risk = stop_loss - entry_price
        if risk > 0:
            position_size = (capital * risk_per_trade) / risk

    # Simulate trade
    if position_size > 0:
        exit_price = row['Close'].item()  # Exit at hour's close
        trade_pnl = 0
        if direction == 'long':
            # Check if stop-loss triggered
            if row['Low'].item() <= stop_loss:
                trade_pnl = (stop_loss - entry_price) * position_size
            else:
                trade_pnl = (exit_price - entry_price) * position_size
        elif direction == 'short':
            if row['High'].item() >= stop_loss:
                trade_pnl = (entry_price - stop_loss) * position_size
            else:
                trade_pnl = (entry_price - exit_price) * position_size

        # Apply fees (entry and exit)
        fees = (entry_price * position_size * commission) + (exit_price * position_size * commission)
        trade_pnl -= fees
        capital += trade_pnl
        equity.append(capital)
        returns.append(trade_pnl / (entry_price * position_size))  # Return per trade
        trades.append({
            'timestamp': row.name,
            'direction': direction,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'stop_loss': stop_loss,
            'pnl': trade_pnl,
            'position_size': position_size
        })

# Calculate metrics
trades_df = pd.DataFrame(trades)
if len(trades) > 0:
    win_rate = len(trades_df[trades_df['pnl'] > 0]) / len(trades) * 100
    net_profit = capital - 500
    returns = np.array(returns)
    sharpe_ratio = (returns.mean() / returns.std() * np.sqrt(252 * 24)) if returns.std() != 0 else 0
    equity_series = pd.Series(equity)
    drawdowns = (equity_series.cummax() - equity_series) / equity_series.cummax() * 100
    max_drawdown = drawdowns.max()
else:
    win_rate = 0
    net_profit = 0
    sharpe_ratio = 0
    max_drawdown = 0

# Print results
print("Backtest Results:")
print(f"Net Profit: ${net_profit:.2f}")
print(f"Final Capital: ${capital:.2f}")
print(f"Return: {(net_profit / 500 * 100):.2f}%")
print(f"Win Rate: {win_rate:.2f}%")
print(f"Total Trades: {len(trades)}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2f}%")

# Save trades to CSV
trades_df.to_csv('trades.csv', index=False)

/tmp/ipython-input-14-753049472.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, interval=interval, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-14-753049472.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, interval=interval, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-14-753049472.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  data_5min['hour'] = data_5min.index.floor('H')
/tmp/ipython-input-14-753049472.py:27: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  first_5min = data_5min.groupby('hour').first()[['High', 'Low']]


Backtest Results:
Net Profit: $-500.01
Final Capital: $-0.01
Return: -100.00%
Win Rate: 0.00%
Total Trades: 116
Sharpe Ratio: -406.40
Max Drawdown: 100.00%
